# Huấn luyện Mô hình Dịch Hán - Việt cổ (MarianMT)

Notebook này chỉ tập trung vào việc **huấn luyện mô hình dịch máy** và **đánh giá chất lượng dịch** sử dụng tập dữ liệu song song sạch thu được sau bước dóng hàng.

## Bước 1: Chuẩn bị Môi trường và Cài đặt Thư viện

In [ ]:
!nvidia-smi
!pip install -q transformers[torch] datasets evaluate sacrebleu accelerate tensorboard pandas openpyxl

## Bước 2: Tải Script Huấn luyện chuẩn của Hugging Face

In [ ]:
!wget -q https://raw.githubusercontent.com/huggingface/transformers/main/examples/pytorch/translation/run_translation.py
!ls -la run_translation.py

## Bước 3: Nạp Dataset của bạn

* Cách 1: Chạy trực tiếp sau bước dóng hàng (đường dẫn `./SinoNom-NLP/output/translation_dataset/`)
* Cách 2: Tải file zip `alignment_output.zip` lên Kaggle, giải nén và trỏ đường dẫn tới đó.

In [ ]:
import os
# Kiểm tra các đường dẫn dữ liệu khả dụng
paths = [
    "output/translation_dataset",
    "SinoNom-NLP/output/translation_dataset",
    "/kaggle/input/sino-nom-translation-dataset/translation_dataset"
]

DATASET_PATH = None
for p in paths:
    if os.path.exists(os.path.join(p, "train.json")):
        DATASET_PATH = p
        break

if DATASET_PATH:
    TRAIN_FILE = os.path.join(DATASET_PATH, "train.json")
    VAL_FILE = os.path.join(DATASET_PATH, "val.json")
    print(f"✅ Tìm thấy Dataset tại: {DATASET_PATH}")
    print(f"  - Train file: {TRAIN_FILE}")
    print(f"  - Val file: {VAL_FILE}")
else:
    raise FileNotFoundError("❌ Không tìm thấy train.json/val.json. Vui lòng chạy dóng hàng trước hoặc tải Dataset lên Kaggle.")

## Bước 4: Khởi chạy Huấn luyện (Fine-tuning 20 Epochs)

## Bước 3.5: Tiền xử lý & Làm sạch Dataset trước khi Huấn luyện

Chạy cell dưới đây để:
1. **Tự động xóa bỏ các chú thích** trong ngoặc đơn `(...)` hoặc ngoặc vuông `[...]` ở câu tiếng Việt (đầu vào chữ Hán không có).
2. **Lọc bỏ các cặp câu lệch dóng hàng** thông qua kiểm tra tỷ lệ độ dài (Length Ratio) giữa Hán và Việt.
3. Xuất ra tập dữ liệu sạch đặt tại `/kaggle/working/cleaned_dataset/` để mô hình học chính xác nhất.

In [ ]:
import json
import re
import os

def clean_sentence_vietnamese(text):
    if not text:
        return ""
    # 1. Loại bỏ các lời chú thích trong dấu ngoặc đơn (...) và ngoặc vuông [...]
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    
    # 2. Loại bỏ các tiền tố giải thích hoặc chú thích của dịch giả
    prefixes = [
        r'^\s*Dịch giả chú\s*:\s*',
        r'^\s*Chú thích\s*:\s*',
        r'^\s*Tục danh\s*:\s*',
        r'^\s*Tục gọi\s*:\s*',
        r'^\s*Tạm dịch\s*:\s*'
    ]
    for p in prefixes:
        text = re.sub(p, '', text, flags=re.IGNORECASE)
        
    # 3. Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_valid_pair(zh, vi):
    if not zh or not vi:
        return False
    
    vi_words = len(vi.split())
    zh_chars = len(zh)
    
    if zh_chars < 2 or vi_words < 2:
        return False
        
    # Lọc tỷ lệ độ dài (Length Ratio) giữa Hán và Việt
    # Ngăn các câu cực ngắn Hán dóng hàng với đoạn cực dài Việt
    ratio = zh_chars / vi_words
    if ratio < 0.15 or ratio > 3.5:
        return False
        
    # Loại trừ trường hợp Hán quá ngắn (dưới 5 ký tự) nhưng Việt quá dài (trên 20 từ)
    if zh_chars < 5 and vi_words > 20:
        return False
        
    return True

def clean_dataset(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    cleaned_data = []
    skipped = 0
    
    for item in data:
        # Đọc cấu trúc format HuggingFace: {"translation": {"zh": ..., "vi": ...}}
        trans = item.get("translation", {})
        zh = str(trans.get("zh", "")).strip()
        vi_raw = str(trans.get("vi", ""))
        
        vi_clean = clean_sentence_vietnamese(vi_raw)
        
        if is_valid_pair(zh, vi_clean):
            cleaned_data.append({"translation": {"zh": zh, "vi": vi_clean}})
        else:
            skipped += 1
            
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=2)
        
    return len(data), len(cleaned_data), skipped

# Tiến hành làm sạch
CLEANED_DIR = "/kaggle/working/cleaned_dataset"
NEW_TRAIN_FILE = os.path.join(CLEANED_DIR, "train.json")
NEW_VAL_FILE = os.path.join(CLEANED_DIR, "val.json")

print("🧹 Đang tiến hành làm sạch và chuẩn hóa dữ liệu song song...")
total_tr, clean_tr, skip_tr = clean_dataset(TRAIN_FILE, NEW_TRAIN_FILE)
total_val, clean_val, skip_val = clean_dataset(VAL_FILE, NEW_VAL_FILE)

print(f"📊 TẬP HUẤN LUYỆN (Train): Dòng ban đầu: {total_tr} -> Dòng sạch: {clean_tr} (Lọc bỏ: {skip_tr})")
print(f"📊 TẬP ĐÁNH GIÁ (Val):   Dòng ban đầu: {total_val} -> Dòng sạch: {clean_val} (Lọc bỏ: {skip_val})")

# Cập nhật biến đường dẫn trỏ sang dữ liệu sạch cho cell huấn luyện tiếp theo
TRAIN_FILE = NEW_TRAIN_FILE
VAL_FILE = NEW_VAL_FILE
print("\n✅ Đã cập nhật xong! Biến TRAIN_FILE và VAL_FILE đã trỏ sang dữ liệu làm sạch.")

In [ ]:
!python run_translation.py \
    --model_name_or_path Helsinki-NLP/opus-mt-zh-vi \
    --source_lang zh \
    --target_lang vi \
    --max_source_length 512 \
    --train_file {TRAIN_FILE} \
    --validation_file {VAL_FILE} \
    --output_dir /kaggle/working/han_viet_translation_model \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --do_train \
    --do_eval \
    --num_train_epochs 20 \
    --learning_rate 5e-5 \
    --warmup_ratio 0.1 \
    --save_total_limit 3 \
    --weight_decay 0.01 \
    --predict_with_generate \
    --eval_strategy epoch \
    --save_strategy epoch

## Bước 5: Trực quan hóa quá trình Huấn luyện (Loss & BLEU)

In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

state_file = '/kaggle/working/han_viet_translation_model/trainer_state.json'
if not os.path.exists(state_file):
    ckpts = sorted(glob.glob('/kaggle/working/han_viet_translation_model/checkpoint-*/trainer_state.json'))
    state_file = ckpts[-1] if ckpts else None

if state_file and os.path.exists(state_file):
    with open(state_file) as f:
        state = json.load(f)

    log_history = state.get('log_history', [])
    eval_logs = [l for l in log_history if 'eval_loss' in l]
    train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]

    epochs_eval = [l['epoch'] for l in eval_logs]
    eval_loss   = [l['eval_loss'] for l in eval_logs]
    eval_bleu   = [l.get('eval_bleu', None) for l in eval_logs]

    epochs_train = [l['epoch'] for l in train_logs]
    train_loss   = [l['loss'] for l in train_logs]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Hình 1. Tiến trình huấn luyện mô hình dịch MarianMT', fontweight='bold', fontsize=14)

    # Loss
    axes[0].plot(epochs_train, train_loss, label='Train Loss', color='#4C72B0', marker='o', markersize=3)
    axes[0].plot(epochs_eval, eval_loss, label='Val Loss', color='#C44E52', marker='s', linewidth=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title('Loss theo Epoch'); axes[0].legend()
    axes[0].grid(alpha=0.3)

    # BLEU
    if any(b is not None for b in eval_bleu):
        bleu_vals = [b for b in eval_bleu if b is not None]
        bleu_epochs = [e for e, b in zip(epochs_eval, eval_bleu) if b is not None]
        axes[1].plot(bleu_epochs, bleu_vals, label='Val BLEU', color='#55A868', marker='D', linewidth=2)
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SacreBLEU')
        axes[1].set_title('BLEU theo Epoch'); axes[1].legend()
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("❌ Không tìm thấy trainer_state.json. Vui lòng kiểm tra lại đường dẫn model.")

## Bước 6: Đánh giá định lượng BLEU và so sánh mô hình Gốc vs Fine-tune

In [ ]:
import torch
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
from evaluate import load as eval_load

device = 'cuda' if torch.cuda.is_available() else 'cpu'

tok_orig = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-zh-vi')
mdl_orig = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-zh-vi').to(device)

model_path = '/kaggle/working/han_viet_translation_model'
tok_ft  = MarianTokenizer.from_pretrained(model_path)
mdl_ft  = MarianMTModel.from_pretrained(model_path).to(device)

def translate(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, num_beams=5, max_length=512)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Đọc tập validation
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

val_data = load_jsonl(VAL_FILE)
bleu_metric = eval_load('sacrebleu')

def evaluate_bleu(model, tokenizer, data, desc=''):
    preds, refs = [], []
    for item in data:
        src = item['translation']['zh']
        tgt = item['translation']['vi']
        pred = translate(model, tokenizer, src)
        preds.append(pred)
        refs.append([tgt])
    score = bleu_metric.compute(predictions=preds, references=refs)
    print(f'{desc}: SacreBLEU = {score["score"]:.4f}')
    return score['score'], preds, refs

print('Đang tính toán BLEU trên tập Validation...')
bleu_orig, preds_orig, refs_val = evaluate_bleu(mdl_orig, tok_orig, val_data, 'Mô hình GỐC')
bleu_ft,   preds_ft,   _        = evaluate_bleu(mdl_ft,   tok_ft,   val_data, 'Sau FINE-TUNE')

df_bleu = pd.DataFrame([
    {'Mô hình': 'Helsinki-NLP/opus-mt-zh-vi (Gốc)', 'SacreBLEU': round(bleu_orig, 4)},
    {'Mô hình': 'opus-mt-zh-vi + Fine-tune (Hán-Việt cổ)', 'SacreBLEU': round(bleu_ft, 4)},
])
from IPython.display import display
display(df_bleu.style.set_caption('Bảng 2. So sánh SacreBLEU trên tập Validation'))

## Bước 7: Dịch thử nghiệm 10 câu mẫu (Benchmark)

In [ ]:
test_sentences = [
    '大南一統志卷之二承天府上',
    '嗣德二年以欽文殿爲經筵之所成，',
    '在京城外之春祿邑，',
    '明命七年建正堂前堂各三間，合為一座，正中祀風伯之神，left雲師，right雷師。',
    '紹治六年新建旗柱，通長七丈六尺5寸，上設望斗，凡朝賀巡幸以此。',
    '城池周二千四百八十七丈三尺六寸，高一丈五尺六寸，厚五丈，甃磚。',
    '又建一太醫所在，京城內東福坊，院判及醫生居焉。',
    '嘉隆三年建成，正楹十三間，前楹十五間。',
    '自安南建長月國，在陳為順化地，黎為順化承宣，均稱重鎮。',
    '廟四圍繚以飄墻，前為門樓，樓前為坊門。',
]

rows = []
for i, sent in enumerate(test_sentences, 1):
    pred_orig = translate(mdl_orig, tok_orig, sent)
    pred_ft   = translate(mdl_ft,   tok_ft,   sent)
    rows.append({'#': i, 'Câu Hán': sent, 'Mô hình gốc': pred_orig, 'Sau fine-tune': pred_ft})

df_benchmark = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 120)
display(df_benchmark.style.set_caption('Bảng 3. Benchmark dịch thử — So sánh mô hình gốc và sau fine-tune'))

## Bước 8: Phân tích Lỗi dịch (Error Analysis)

In [ ]:
from sacrebleu.metrics import BLEU as SacreBLEUMetric

sent_bleus = []
bleu_sent_metric = SacreBLEUMetric(effective_order=True)
for pred, ref_list in zip(preds_ft, refs_val):
    score = bleu_sent_metric.sentence_score(pred, ref_list).score
    sent_bleus.append(score)

idx_worst = sorted(range(len(sent_bleus)), key=lambda i: sent_bleus[i])[:5]
error_rows = []
for rank, i in enumerate(idx_worst, 1):
    error_rows.append({
        'Rank': rank,
        'BLEU câu': round(sent_bleus[i], 4),
        'Câu Hán nguồn': val_data[i]['translation']['zh'],
        'Tham chiếu (Ground Truth)': val_data[i]['translation']['vi'],
        'Mô hình dự đoán': preds_ft[i]
    })

df_errors = pd.DataFrame(error_rows)
pd.set_option('display.max_colwidth', 200)
display(df_errors.style.set_caption('Bảng 4. Phân tích lỗi — 5 câu dịch kém nhất của mô hình fine-tune'))

## Bước 9: Nén Mô hình và Tải về

In [ ]:
!zip -q -r trained_model.zip /kaggle/working/han_viet_translation_model/
print('✅ trained_model.zip — Mô hình đã fine-tune hoàn tất để tải về.')